In [1]:
!pip install category_encoders
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 2.1 MB/s eta 0:00:00


In [2]:
import joblib
import pandas as pd
import numpy as np
import category_encoders as ce
from sklearn.base import BaseEstimator, TransformerMixin

In [3]:
class ReplaceMinusOneWithNanTransformer(BaseEstimator, TransformerMixin):
  def __init__(self, columns):
        self.columns = columns
  def fit(self, X, y=None): return self
  def transform(self, X):
      X_ = X.copy()
      for col in self.columns:
          X_[col] = X_[col].replace(-1, np.nan)
      return X_

In [4]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
  def __init__(self, current_year=1404):
    self.current_year = current_year
    self.location_max_floor_ = None

  def fit(self, X, y=None):
    X_ = X.copy()
    self.feature_names_in_ = X.columns
    X_['floor'] = X_['floor'].replace(-1, np.nan)
    self.location_max_floor_ = X_.groupby('location')['floor'].max()
    return self

  def transform(self, X):
    X_ = X.copy()
    # Basic Features
    X_['age'] = self.current_year - X_['build_year']
    X_['age'] = X_['age'].apply(lambda x: x if 0 <= x <= 50 else np.nan)

    # Handle division by zero for rooms
    X_['room_density'] = X_['meterage'] / X_['rooms'].replace(0, 1)

    # Feature: floor_is_top
    X_['floor_is_top'] = (X_['floor'] == X_['location'].map(self.location_max_floor_)).astype(int)

    # Feature: age_bucket
    age_bins = [-1, 5, 15, 30, 200]
    age_labels = ['0-5', '6-15', '16-30', '30+']
    X_['age_bucket'] = pd.cut(X_['age'], bins=age_bins, labels=age_labels, right=True)

    # Feature: rooms_per_floor
    X_['rooms_per_floor'] = X_['rooms'] / (X_['floor'] + 1)

    # Feature: density_level
    try:
        density_bins = pd.qcut(X_['room_density'], q=4, labels=False, duplicates='drop')
        density_labels = ['low', 'medium', 'high', 'very_high']
        X_['density_level'] = pd.cut(X_['room_density'], bins=pd.qcut(X_['room_density'], q=4, retbins=True, duplicates='drop')[1], labels=density_labels, include_lowest=True)
    except ValueError: # Handle cases with not enough unique values for qcut
        X_['density_level'] = 'medium'

    # Make rare locations 'Other'
    location_counts = X_['location'].value_counts()
    rare_locations = location_counts[location_counts <= 20].index
    X_['location'] = X_['location'].replace(rare_locations, 'Other')

    return X_

In [5]:
class OutlierCapperIQR(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns
        self.bounds_ = {}

    def fit(self, X, y=None):
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.columns)

        for col in self.columns:
            Q1 = X_df[col].quantile(0.25)
            Q3 = X_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            self.bounds_[col] = (lower_bound, upper_bound)

        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy_df = X_copy if isinstance(X_copy, pd.DataFrame) else pd.DataFrame(X_copy, columns=self.columns)

        for col in self.columns:
            lower, upper = self.bounds_[col]
            X_copy_df[col] = X_copy_df[col].clip(lower=lower, upper=upper)

        return X_copy_df

    def get_feature_names_out(self, input_features=None):
        return input_features

In [6]:
class LocationFeaturesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, north=None, south=None, east=None, west=None, center=None):
        self.north = north or []
        self.south = south or []
        self.east = east or []
        self.west = west or []
        self.center = center or []
        self.region_encoder = ce.TargetEncoder(smoothing=5)
        self.district_encoder = ce.TargetEncoder(smoothing=5)

    def fit(self, X, y=None):
        X_ = X.copy()
        X_['district'] = X_['location'].apply(self._extract_district)
        X_['region'] = X_['district'].apply(self._map_region)

        self.region_encoder.fit(X_['region'], y)
        self.district_encoder.fit(X_['district'], y)
        return self

    def transform(self, X):
        X_ = X.copy()
        X_['district'] = X_['location'].apply(self._extract_district)
        X_['region'] = X_['district'].apply(self._map_region)

        region_encoded = self.region_encoder.transform(X_['region'])
        district_encoded = self.district_encoder.transform(X_['district'])

        result = pd.concat([X_.drop(['location', 'district', 'region'], axis=1), region_encoded, district_encoded], axis=1)
        result.columns = X_.drop(['location', 'district', 'region'], axis=1).columns.tolist() + ['region_encoded', 'district_encoded']
        return result

    def _extract_district(self, loc):
        try:
            return loc.split('،')[1].strip()
        except:
            return 'Other'

    def _map_region(self, district):
        if district in self.north: return 'north'
        if district in self.south: return 'south'
        if district in self.east: return 'east'
        if district in self.west: return 'west'
        if district in self.center: return 'center'
        return 'unknown'

In [8]:
print("Loading prediction components...")
base_pipeline = joblib.load("base_pipeline.pkl")
location_encoder = joblib.load("location_encoder.pkl")
base_feature_names = joblib.load("base_feature_names.pkl")
# rf_model = joblib.load("rf_model.pkl")
xgb_model = joblib.load("best_xgb_model.pkl")
print("Components loaded.")

Loading prediction components...
Components loaded.


In [13]:
def predict_price(data_df):
    """
    Takes a DataFrame with a single row of new house data and predicts its price.
    """
    print("\nStarting transformation...")
    # Step 1: Apply the base pipeline
    processed_base_array = base_pipeline.transform(data_df)
    print(f"Step 1 (base pipeline) complete. Shape: {processed_base_array.shape}")

    # Step 2: Convert to DataFrame and add back the original location
    processed_df = pd.DataFrame(processed_base_array, columns=base_feature_names, index=data_df.index)
    df_for_loc_encoding = processed_df.join(data_df['location'])
    print("Step 2 (adding location) complete.")

    # Step 3: Apply the location encoder
    final_prepared_df = location_encoder.transform(df_for_loc_encoding)
    print(f"Step 3 (location encoder) complete. Final shape: {final_prepared_df.shape}")

    # Ensure the final columns match what the model was trained on
    # This is a safety check.
    if final_prepared_df.shape[1] != 25:
        print(f"Error: Expected 25 features, but got {final_prepared_df.shape[1]}")
        return None

    # Ensure correct data types for XGBoost
    for col in final_prepared_df.columns:
        if final_prepared_df[col].dtype == 'object':
            try:
                # Attempt to convert to numeric, coercing errors to NaN
                final_prepared_df[col] = pd.to_numeric(final_prepared_df[col], errors='coerce')
            except:
                # If numeric conversion fails, try converting to boolean
                try:
                    final_prepared_df[col] = final_prepared_df[col].astype(bool)
                except:
                    # Handle other object types if necessary, or leave as NaN/None
                    print(f"Warning: Could not convert column '{col}' to numeric or boolean. Leaving as is.")


    # Step 4: Predict using the ensemble models
    print("Predicting with ensemble model...")
    # pred_rf_log = rf_model.predict(final_prepared_df)
    pred_xgb_log = xgb_model.predict(final_prepared_df)

    # Step 5: Blend the predictions
    # ensemble_pred_log = (pred_rf_log + pred_xgb_log) / 2

    # Step 6: Inverse transform to get the final price
    final_price_billion = np.expm1(pred_xgb_log)

    return final_price_billion[0]

In [22]:
test_data = {
    "meterage": 130.0,
    "build_year": 1403.0,
    "rooms": 3.0,
    "floor": 19.0,
    "has_parking": True,
    "has_warehouse": True,
    "has_balcony": False,
    "has_elevator": True,
    "is_renovated": False,
    "location": 'تهران، دریاچه شهدای خلیج فارس'
}

In [23]:
user_df = pd.DataFrame([test_data])

In [24]:
predicted_price = predict_price(user_df)


Starting transformation...
Step 1 (base pipeline) complete. Shape: (1, 23)
Step 2 (adding location) complete.
Step 3 (location encoder) complete. Final shape: (1, 25)
Predicting with ensemble model...


In [25]:
user_df.head()

,meterage,build_year,rooms,floor,has_parking,has_warehouse,has_balcony,has_elevator,is_renovated,location
0,130.0,1403.0,3.0,19.0,True,True,False,True,False,تهران، دریاچه شهدای خلیج فارس


In [26]:
if predicted_price is not None:
    print("\n-------------------------------------------")
    print(f"Predicted Price: {predicted_price:.2f} Billion Toman")
    print("-------------------------------------------")



-------------------------------------------
Predicted Price: 3.43 Billion Toman
-------------------------------------------
